# Notebook 06 — Visualising Attention

**Goal:** Load the trained model and inspect what the attention heads learned.

---

### Contents
1. [Load the model](#1)
2. [Per-head attention heatmaps](#2)
3. [Average attention per layer](#3)
4. [Attention rollout](#4)
5. [Try your own text](#5)
6. [Key takeaways](#6)


In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, sys
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import tiktoken

sys.path.insert(0, os.path.abspath('.'))
from utils.model import GPTModel
from utils.visualisation import compute_attention_rollout

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('=' * 60)
print('  Notebook 06 — Visualising Attention')
print('=' * 60)


  Notebook 06 — Visualising Attention


<a id='1'></a>
## 1 — Load the model


In [2]:
checkpoint_path = 'checkpoints/model.pt'
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError('Run notebook 05 first to create checkpoints/model.pt')

ckpt   = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
config = ckpt['config']
model  = GPTModel(config)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

enc = tiktoken.get_encoding('gpt2')

print(f'Checkpoint : {checkpoint_path}')
print(f'Layers     : {config["n_layers"]}')
print(f'Heads      : {config["n_heads"]}')
print(f'Parameters : {model.count_parameters():,}')


FileNotFoundError: Run notebook 05 first to create checkpoints/model.pt

In [ ]:
def run_with_attention(text):
    """Forward pass → returns (token_strings, list_of_attn_per_layer)."""
    ids    = enc.encode(text)
    tokens = [enc.decode([t]) for t in ids]
    with torch.no_grad():
        _ = model(torch.tensor([ids], dtype=torch.long))
    # Each element: (n_heads, seq, seq)
    attn = [w.squeeze(0) for w in model.get_attention_weights()]
    return tokens, attn


text = 'A transformer uses self attention to process sequences'
tokens, attn_layers = run_with_attention(text)
print(f'Tokens ({len(tokens)}): {tokens}')


<a id='2'></a>
## 2 — Per-head attention heatmaps

The most direct way to see what each head is doing.


In [ ]:
for layer_idx, layer_attn in enumerate(attn_layers):
    n_heads = layer_attn.shape[0]
    fig, axes = plt.subplots(1, n_heads, figsize=(4 * n_heads, 4))
    if n_heads == 1:
        axes = [axes]

    for h, ax in enumerate(axes):
        sns.heatmap(layer_attn[h].numpy(),
                    annot=len(tokens) <= 8, fmt='.2f',
                    cmap='Blues', vmin=0, vmax=1,
                    xticklabels=tokens, yticklabels=tokens,
                    linewidths=0.3, linecolor='white',
                    square=True, ax=ax)
        ax.set_title(f'L{layer_idx} H{h}', fontweight='bold', fontsize=10)
        ax.set_xlabel('Key')
        ax.set_ylabel('Query')

    plt.suptitle(f'Layer {layer_idx}', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


<a id='3'></a>
## 3 — Average attention per layer

Averaging over heads gives a simpler view of how a layer distributes attention.


In [ ]:
fig, axes = plt.subplots(1, len(attn_layers), figsize=(5 * len(attn_layers), 4))
if len(attn_layers) == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    avg = attn_layers[i].mean(dim=0).numpy()
    sns.heatmap(avg, annot=len(tokens) <= 8, fmt='.2f',
                cmap='Blues', vmin=0,
                xticklabels=tokens, yticklabels=tokens,
                linewidths=0.3, linecolor='white',
                square=True, ax=ax)
    ax.set_title(f'Layer {i} (avg over heads)', fontweight='bold')
    ax.set_xlabel('Key')
    ax.set_ylabel('Query')

plt.tight_layout()
plt.show()


<a id='4'></a>
## 4 — Attention rollout

Raw attention is per-layer.  **Rollout** propagates attention through all layers
to estimate the effective information flow from input tokens to output positions.


In [ ]:
rollout = compute_attention_rollout(attn_layers)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(rollout, annot=len(tokens) <= 8, fmt='.2f',
            cmap='viridis', vmin=0,
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.3, linecolor='white',
            square=True, ax=ax)
ax.set_title('Attention Rollout (all layers combined)', fontweight='bold')
ax.set_xlabel('Input token')
ax.set_ylabel('Output position')
plt.tight_layout()
plt.show()

print('Rollout shows cumulative information flow, not just single-layer attention.')


<a id='5'></a>
## 5 — Try your own text

Change `your_text` below and re-run the cell.


In [ ]:
# ── Edit this and re-run ──────────────────────────────────────────────────────
your_text = 'Engineering systems use feedback to control behaviour'

your_tokens, your_attn = run_with_attention(your_text)
print(f'Tokens: {your_tokens}')

avg = your_attn[0].mean(dim=0).numpy()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(avg, annot=len(your_tokens) <= 8, fmt='.2f',
            cmap='Blues', vmin=0,
            xticklabels=your_tokens, yticklabels=your_tokens,
            linewidths=0.3, linecolor='white',
            square=True, ax=ax)
ax.set_title('Layer 0 average attention', fontweight='bold')
ax.set_xlabel('Key')
ax.set_ylabel('Query')
plt.tight_layout()
plt.show()


<a id='6'></a>
## 6 — Key takeaways

- **Per-head heatmaps** are the clearest way to inspect what each head learned
- **Average over heads** gives a simpler layer-level summary
- **Attention rollout** estimates cumulative information flow across all layers
- These three views cover most of what you need for quick attention analysis

### 🎉 Done!

You've built a transformer from scratch, trained it, and visualised its internals.
The same architecture — just scaled up — powers GPT-4, Claude, and every modern LLM.
